# 05 Silver → Gold: Merging Sensor Streams Into One Gold Table

## Purpose

This notebook reads the **cleaned sensor data** (parquet files) from the silver layer in Azure, merges all domains for each city, and produces a single **gold table** (comprehensive data warehouse fact table) ready for analytics.

**Designed for:** Azure Synapse Notebooks (credentials available from environment)

**Data flow:**
```
silver/ (Azure ABFSS)
  ├── schwartau/
  │   ├── flow/            → (latest .parquet)
  │   ├── temperature/     → (latest .parquet)
  │   ├── humidity/        → (latest .parquet)
  │   └── weight/          → (latest .parquet)
  └── [other cities]/
  
            ↓ [merge by timestamp]
            
gold/ (Azure ABFSS)
  └── hivedata__{timestamp}.parquet    ← one comprehensive table
```

## Key Assumptions

1. **Cloud Storage:** All data is stored in Azure ABFSS (Azure Data Lake Storage Gen 2).
2. **Structure:** Each city has subfolders (domains) under `silver/{city}/`, with one `.parquet` file per domain.
3. **Latest wins:** We read the **most recent** parquet file from each domain (by filename alphabetical order).
4. **All cities:** The notebook processes ALL cities found in the silver folder automatically.
5. **Timestamp join:** All domains have a `timestamp` column (in UTC). We merge on this column using **outer join** (preserve all timestamps).
6. **Column collisions:** Weather data has `temperature` and `humidity` columns. Sensor versions are renamed to `temperature_hive` and `humidity_hive` to avoid conflicts.

## Getting Started

To run this notebook:
1. **Fill in credentials** in the Setup cell (ACCOUNT_NAME, ACCOUNT_KEY) if running locally
2. **Or** run in Azure Synapse where credentials are automatically available
3. Update `<container>` and `<account>` in the SILVER_ROOT and GOLD_ROOT paths

## What You Will Learn

- How to discover and load the **latest parquet files** from nested folder structures in Azure
- How to **merge multiple data streams** on a common timestamp (handling sparse/missing data)
- How to **handle column name collisions** gracefully
- How to **preserve column order** for presentation to business teams


In [1]:
import fsspec, pandas as pd
from datetime import datetime, timezone

# Azure Synapse paths (cloud storage only)
SILVER_ROOT = "abfss://<container>@<account>.dfs.core.windows.net/silver"
GOLD_ROOT = "abfss://<container>@<account>.dfs.core.windows.net/gold"

print(f"Silver: {SILVER_ROOT}\nGold: {GOLD_ROOT}")

Silver: abfss://<container>@<account>.dfs.core.windows.net/silver
Gold: abfss://<container>@<account>.dfs.core.windows.net/gold


In [4]:
import fsspec
import pandas as pd
from datetime import datetime, timezone

# ────────────────────────────────────────────────────────────────────────────
# SETUP: Define Azure paths
# ────────────────────────────────────────────────────────────────────────────

# In Azure Synapse, credentials are automatically available from the environment.
# For local testing, fill in your account name and key below.

ACCOUNT_NAME  = None  # Set to your storage account name if testing locally
ACCOUNT_KEY   = None  # Set to your account key if testing locally

SILVER_ROOT = "abfss://<container>@<account>.dfs.core.windows.net/silver"
GOLD_ROOT   = "abfss://<container>@<account>.dfs.core.windows.net/gold"

print(f"Silver root: {SILVER_ROOT}")
print(f"Gold root:   {GOLD_ROOT}")
print("✓ Paths configured")

# Helper function to get filesystem — handles credentials from environment
def get_fs():
    if ACCOUNT_NAME and ACCOUNT_KEY:
        return fsspec.filesystem("abfss", account_name=ACCOUNT_NAME, account_key=ACCOUNT_KEY)
    else:
        # In Synapse, credentials come from environment automatically
        return fsspec.filesystem("abfss")


Silver root: abfss://<container>@<account>.dfs.core.windows.net/silver
Gold root:   abfss://<container>@<account>.dfs.core.windows.net/gold
✓ Paths configured


---

## Step 1a — Discover All Cities

List all city folders in the silver layer. The notebook processes **all cities found here automatically**.


In [5]:
# List all folders in the silver root — each one is a city
fs = get_fs()
entries = fs.ls(SILVER_ROOT, detail=True)
cities = sorted(set([e["name"].split("/")[-1] for e in entries if e.get("type") == "directory"]))

print(f"🌍 Found {len(cities)} city(ies):")
for city in cities:
    print(f"   • {city}")


ValueError: unable to connect to account for Must provide either a connection_string or account_name with credentials!!

---

## Step 1b — Load One City (All Its Domains)

For each city, we load the **latest parquet file** from each domain folder:
- `silver/{city}/flow/`
- `silver/{city}/temperature/`
- `silver/{city}/humidity/`
- `silver/{city}/weight/`
- `silver/{city}/weather/`

Then we **merge them all on the timestamp column** (outer join, so no data is lost).


In [6]:
def load_city(city_name, silver_root):
    """Load all sensor domains for one city and merge them into a single table."""
    
    fs = get_fs()  # Get filesystem object
    
    city_folder = f"{silver_root}/{city_name}"
    domains = ["flow", "temperature", "humidity", "weight", "weather"]
    datasets = {}
    
    print(f"\n📁 {city_name}:")
    
    # ────────────────────────────────────────────────────────────────────────
    # STEP 1: Load each domain's latest parquet file
    # ────────────────────────────────────────────────────────────────────────
    
    for domain in domains:
        domain_folder = f"{city_folder}/{domain}"
        
        # Find all parquet files in this domain folder
        parquet_files = sorted(fs.glob(f"{domain_folder}/*.parquet"))
        if not parquet_files:
            continue  # This domain doesn't exist for this city
        
        # Read the LAST (most recent) parquet file
        latest_file = parquet_files[-1]
        df = pd.read_parquet(latest_file, filesystem=fs)
        
        # Skip if timestamp column is missing
        if "timestamp" not in df.columns:
            continue
        
        # ────────────────────────────────────────────────────────────────────
        # STEP 2: Parse timestamps to UTC
        # ────────────────────────────────────────────────────────────────────
        
        df = df.copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
        df = df.dropna(subset=["timestamp"])
        
        # ────────────────────────────────────────────────────────────────────
        # STEP 3: Handle column name collisions
        # ────────────────────────────────────────────────────────────────────
        # Weather data has "temperature" and "humidity" columns.
        # Rename sensor versions to avoid collisions when merging.
        
        if domain == "temperature" and "temperature" in df.columns:
            df = df.rename(columns={"temperature": "temperature_hive"})
        if domain == "humidity" and "humidity" in df.columns:
            df = df.rename(columns={"humidity": "humidity_hive"})
        
        datasets[domain] = df
        print(f"  ✓ {domain:12s}  {len(df):>10,} rows")
    
    if not datasets:
        print(f"  ⚠️  No data found")
        return pd.DataFrame()
    
    # ────────────────────────────────────────────────────────────────────────
    # STEP 4: Merge all domains on timestamp (outer join)
    # ────────────────────────────────────────────────────────────────────────
    # Outer join means: keep ALL timestamps, even if some sensors are missing
    
    print(f"  Merging domains...")
    merged = None
    for domain in domains:
        if domain in datasets:
            if merged is None:
                merged = datasets[domain].copy()
            else:
                merged = merged.merge(datasets[domain], on="timestamp", how="outer")
    
    # ────────────────────────────────────────────────────────────────────────
    # STEP 5: Clean up and add city tag
    # ────────────────────────────────────────────────────────────────────────
    
    merged = merged.sort_values("timestamp").reset_index(drop=True)
    merged = merged.drop_duplicates(subset=["timestamp"], keep="last")
    merged["location"] = city_name
    
    return merged


# ═══════════════════════════════════════════════════════════════════════════
# TEST: Load one city to verify the logic works
# ═══════════════════════════════════════════════════════════════════════════

test_city = load_city("schwartau", SILVER_ROOT)
print(f"\n✓ Loaded {len(test_city):,} rows × {len(test_city.columns)} columns")
if len(test_city.columns) > 0:
    print(f"  Columns: {list(test_city.columns)}")


ValueError: unable to connect to account for Must provide either a connection_string or account_name with credentials!!

---

## Step 2 — Build Gold Table (Merge All Cities)

Now we use the `load_city()` function on **every discovered city** and stack them all into one big gold table.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Build the gold table: loop through all cities and merge them
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print(f"🏆 Processing {len(cities)} city(ies): {cities}")
print("="*70)

# ────────────────────────────────────────────────────────────────────────────
# STEP 1: Load each city
# ────────────────────────────────────────────────────────────────────────────

city_tables = []
for city in cities:
    df = load_city(city, SILVER_ROOT)
    if not df.empty:
        city_tables.append(df)

if not city_tables:
    raise ValueError("❌ No data found in any city!")

print(f"\n✓ Loaded {len(city_tables)} cities")

# ────────────────────────────────────────────────────────────────────────────
# STEP 2: Concatenate all cities into one big table
# ────────────────────────────────────────────────────────────────────────────

print(f"📊 Concatenating all cities...")
gold = pd.concat(city_tables, ignore_index=True)

print(f"  → {len(gold):,} total rows")

# ────────────────────────────────────────────────────────────────────────────
# STEP 3: Optimize memory by converting location to category
# ────────────────────────────────────────────────────────────────────────────

gold["location"] = gold["location"].astype("category")

# ────────────────────────────────────────────────────────────────────────────
# STEP 4: Reorder columns (business team wants important columns first)
# ────────────────────────────────────────────────────────────────────────────

preferred_order = [
    "timestamp", "location",
    "flow_out", "flow_in", "flow_total",
    "temperature_hive", "humidity_hive", "weight",
    "precipitation", "pressure_msl", "sunshine", "temperature",
    "wind_direction", "wind_speed", "cloud_cover", "dew_point",
    "relative_humidity", "wind_gust_direction", "wind_gust_speed",
    "solar", "station_id", "distance", "station_name"
]

# Keep only columns that exist, in preferred order
cols_present = [c for c in preferred_order if c in gold.columns]
cols_extra = [c for c in gold.columns if c not in cols_present]
gold = gold[cols_present + cols_extra]

# ────────────────────────────────────────────────────────────────────────────
# STEP 5: Final sort by city, then timestamp
# ────────────────────────────────────────────────────────────────────────────

gold = gold.sort_values(["location", "timestamp"]).reset_index(drop=True)

print(f"✓ Gold table ready")


✓ Build function ready


---

## Step 3 — Save to Azure Storage

Write the gold table to a parquet file with a timestamp in the filename.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# SAVE: Write gold table to parquet file
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("💾 SAVING GOLD TABLE")
print("="*70)

# Create timestamp for the filename
timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%dT%Hh%Mm%Ss')
output_filename = f"hivedata__{timestamp}.parquet"
output_path = f"{GOLD_ROOT}/{output_filename}"

print(f"\nWriting to: {output_path}")

# Get filesystem and write to parquet
fs = get_fs()
with fs.open(output_path, "wb") as f:
    gold.to_parquet(f, index=False)

# ════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════════════════════════

print(f"\n✅ SUCCESS!")
print(f"   Rows:       {len(gold):,}")
print(f"   Columns:    {len(gold.columns)}")
print(f"   Size:       {gold.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"   Cities:     {list(gold['location'].unique())}")
print(f"   Timestamps: {gold['timestamp'].min()} → {gold['timestamp'].max()}")
print()

# Show a preview
gold.head(10)



🏆 BUILDING GOLD LAYER

🌍 Processing 2 city(ies): ['processing', 'schwartau']

📁 Loading processing:
  ⚠️  No data found

📁 Loading schwartau:
  ✓ flow           1,267,141 rows
  ✓ temperature      253,430 rows
  ✓ humidity           1,761 rows
  ✓ weight             1,761 rows
  ✓ weather           21,121 rows
  Merging domains...

📊 Concatenating 1 cities...

💾 Saving...

✅ SUCCESS!
   Rows: 1,267,156
   Columns: 23
   Size: 244.7 MB
   Cities: ['schwartau']
   Time range: 2017-01-01 00:00:00+00:00 to 2019-05-31 12:15:00+00:00
   Saved to: ../data/gold/hivedata__2026-02-19T09h41m09s.parquet



,timestamp,location,flow_out,flow_in,flow_total,temperature_hive,humidity_hive,weight,precipitation,pressure_msl,...,wind_speed,cloud_cover,dew_point,relative_humidity,wind_gust_direction,wind_gust_speed,solar,station_id,distance,station_name
0,2017-01-01 00:00:00+00:00,schwartau,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,286541.0,2122.0,"Schwartau,Bad -Groß Parin"
1,2017-01-01 01:00:00+00:00,schwartau,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,286541.0,2122.0,"Schwartau,Bad -Groß Parin"
2,2017-01-01 02:00:00+00:00,schwartau,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,286541.0,2122.0,"Schwartau,Bad -Groß Parin"
3,2017-01-01 03:00:00+00:00,schwartau,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1018.1,...,7.6,100.0,3.0,93.0,210.0,19.8,0.0,7102.0,13018.0,Lübeck-Blankensee
4,2017-01-01 04:00:00+00:00,schwartau,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1017.2,...,7.9,100.0,3.1,92.0,250.0,22.3,0.0,7102.0,13018.0,Lübeck-Blankensee
5,2017-01-01 05:00:00+00:00,schwartau,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1016.4,...,9.0,100.0,3.2,92.0,210.0,20.2,0.0,7102.0,13018.0,Lübeck-Blankensee
6,2017-01-01 06:00:00+00:00,schwartau,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1016.0,...,9.7,100.0,3.3,92.0,230.0,26.3,0.0,7102.0,13018.0,Lübeck-Blankensee
7,2017-01-01 07:00:00+00:00,schwartau,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1015.3,...,10.4,87.0,3.5,93.0,250.0,27.0,0.0,7102.0,13018.0,Lübeck-Blankensee
8,2017-01-01 08:00:00+00:00,schwartau,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1015.4,...,8.6,100.0,3.4,92.0,220.0,22.3,0.0,7102.0,13018.0,Lübeck-Blankensee
9,2017-01-01 09:00:00+00:00,schwartau,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1015.0,...,10.4,100.0,3.1,91.0,210.0,30.2,0.0,7102.0,13018.0,Lübeck-Blankensee
